## Chapter 1 · Understand the dataset and the inspection process

**Before the first lesson:** Extract the complete pack and open its folder in VS Code.
Use the course Python 3.14 environment with the Python and Jupyter extensions, then select
that environment as this notebook's kernel. Install the packages once:

```bash
python -m pip install "pandas==3.0.5" openpyxl ipykernel
```

Pandas supplies the table methods, `openpyxl` writes the final Excel report, and `ipykernel`
runs Python in a notebook. `pathlib`, which supplies file paths, is included with Python.
If the terminal uses a different interpreter, `%pip install "pandas==3.0.5" openpyxl ipykernel`
in a temporary notebook cell installs into the selected kernel. Restart the kernel after installation.
The README contains the chapter order and folder layout.


## Block 1 · Begin with questions, not commands

**Objective:** Understand what we want the data to help us answer.

We are preparing a report about reported wildfires. Inspection asks whether the information
can support the report. It does not yet clean the data or answer all the analysis questions.

| Question | Information needed | Why inspection matters |
| --- | --- | --- |
| **Q1:** How do recorded fire counts compare across years and states? | `fire_year`, `state`, one record per row | Unsuitable years, codes or repeated records can change the groups and counts. |
| **Q2:** How does recorded fire size compare across reported causes? | `fire_size`, `fire_cause` | A readable number may still be unsuitable; an undefined cause cannot identify a reported cause. |
| **Q3:** How long did fires take from discovery to containment? | Both date fields | Missing dates or reversed dates limit the calculation. |
| **Q4:** Which names can we show when discussing particular records? | `fire_name`, supporting fields | Missing names and formatting differences make identification harder. |

**About the source:** The teaching CSV is an eight-field extract associated with the
[fourth-edition FPA fire-occurrence dataset](https://www.fs.usda.gov/rds/archive/catalog/RDS-2013-0009.4) for 1992–2015. Its original source is
Short, Karen C. (2017), *Spatial wildfire occurrence data for the United States, 1992–2015*,
DOI [10.2737/RDS-2013-0009.4](https://doi.org/10.2737/RDS-2013-0009.4).
The CSV headings below are teaching names; they are not all the original database headings.

| Teaching field | Original field | Plain-English meaning |
| --- | --- | --- |
| `fire_name` | `FIRE_NAME` | Reported fire name; may be absent. |
| `state` | `STATE` | Reported state or jurisdiction code. |
| `discovery_date` | `DISCOVERY_DATE` | Date the fire was discovered or confirmed. |
| `contained_date` | `CONTAIN_DATE` | Date it was declared contained; may be absent. |
| `fire_year` | `FIRE_YEAR` | Reported calendar year of discovery. |
| `fire_size` | `FIRE_SIZE` | Estimated area in acres within the final perimeter. |
| `fire_class` | `FIRE_SIZE_CLASS` | Reported size class, A–G; detailed boundaries are deferred. |
| `fire_cause` | `STAT_CAUSE_DESCR` | Reported cause label; `Missing/Undefined` is a source label. |

**One source row is one reported record.** This extract has no incident identifier.
A repeated name does not establish that two records describe the same fire. We can identify
rows that need review, but these eight fields alone may not establish physical-fire identity.

**Pause and explain:** A fire has no name but has a size and cause. Which questions can it still help answer?
**Suggested answer:** It may still help with Q1 and Q2. Missing a name limits Q4; it does not make every field unusable.


## Block 2 · Load and preview the original table

**Objective:** Check that the intended file has loaded as a table.

**Expectation:** The teaching file has the eight headings listed above and at least one record.

**Where it comes from:** The supplied teaching-file description. An unexpected import should be resolved before value checks.

**Objects used here:** `raw` will be the original DataFrame. One row means one imported record.

**Method reference:** [read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [head()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html) · [columns](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.columns.html) · [len()](https://docs.python.org/3/library/functions.html#len) · [Path](https://docs.python.org/3/library/pathlib.html#pathlib.Path)


In [ ]:
import pandas as pd
from pathlib import Path


# Start from the same unchanged teaching file in every notebook.
# Paths are relative to the study-guide folder opened in VS Code.
DATASET_PATH = Path("Data/wildfires.csv")
raw = pd.read_csv(DATASET_PATH, low_memory=False)

# An empty table cannot show that values passed a check.
if len(raw) == 0:
    raise ValueError("No records were loaded. Check Data/wildfires.csv before continuing.")

print("Source file:", DATASET_PATH)
print("Imported records:", len(raw))
print("Columns:", raw.columns.tolist())


In [ ]:
# Reading a few rows helps us connect the field descriptions to actual values.
# It does not tell us how common a problem is in the whole file.
display(raw.head(5))


**Read the result:** The count describes all loaded records. The displayed table is only five examples. The supplied source has 1,880,465 records and eight columns; use your computed dimensions if your file differs.

**Next action:** If the headings differ, confirm the file and delimiter before continuing.

**Pause and explain:** Why can five displayed rows not establish that the whole file has no missing dates?

<details>
<summary>Suggested answer — open after discussing</summary>

Other rows may have missing dates. We need a full-column check to measure that condition.

</details>


## Block 3 · Recognise the objects before using them

**Objective:** Distinguish a table, a column, a True/False decision and one count.

**Expectation:** A Boolean Series contains one True/False decision for each labelled item.

**Where it comes from:** Pandas uses labelled tables and columns. A row label connects a condition to the source row it describes.

**Objects used here:** `object_demo` is a new three-record demonstration. It is separate from `raw`.

**Method reference:** [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [Series()](https://pandas.pydata.org/docs/reference/api/pandas.Series.html) · [isna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html) · [sum()](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html) · [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.index.html)


In [ ]:
# A small demonstration makes every decision visible. These are invented examples.
object_demo = pd.DataFrame({
    "fire_name": ["Oak", None, "Pine"],
    "fire_size": [3.0, 5.0, 1.0]
})
name_column = object_demo["fire_name"]  # One Series value belongs to each record.
missing_name_demo = name_column.isna()  # True means the name is absent in that row.

# True contributes 1 and False contributes 0. The sum counts matching records.
missing_name_demo_count = missing_name_demo.sum()
print("True/False decisions:")
display(missing_name_demo)
print("Matching records:", missing_name_demo_count)

# loc uses those row decisions to select the original demonstration records.
# Their row labels are retained so we can trace each selected record.
selected_demo_records = object_demo.loc[missing_name_demo, ["fire_name", "fire_size"]]
display(selected_demo_records)


**Read the result:** The DataFrame has three records; the name Series has three values; the condition has three decisions; the count is the single number 1. The selected table has the record labelled 1.



**Pause and explain:** Does the row label 1 identify a particular physical fire?

<details>
<summary>Suggested answer — open after discussing</summary>

It identifies a row in this demonstration import. It is not an incident identifier.

</details>


## Block 4 · Inspect structure and storage types

**Objective:** Understand the loaded table before judging its individual values.

**Expectation:** A storage type describes how Pandas stores values, not whether their meanings are correct.

**Where it comes from:** Import behaviour and the field descriptions in Block 1.

**Objects used here:** `raw` is the original eight-field source table loaded in Block 2. `raw.shape` supplies row and column counts.

**Method reference:** [shape](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html) · [dtypes](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dtypes.html) · [info()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html)


In [ ]:
# Dimensions help confirm that the import separated fields as expected.
print("Rows and columns:", raw.shape)
display(raw.dtypes)

# Request the non-missing counts explicitly because this is a large table.
# Date-looking text is still text until we interpret it as a date in Chapter 3.
raw.info(show_counts=True)


**Read the result:** Compare each type and non-missing count with the field meaning. A numerical type permits arithmetic but does not prove a positive area. A date-looking text column needs interpretation.



**Pause and explain:** Can a negative fire size have a numerical storage type?

<details>
<summary>Suggested answer — open after discussing</summary>

Yes. Storage type and sensible meaning are different questions.

</details>


## Block 5 · Know where rules and check references come from

**Objective:** Explain an expectation before writing its test.

| Term | Meaning in this guide | Example |
| --- | --- | --- |
| **Rule / expectation** | What we expect, with a reason. | Containment should not precede discovery. |
| **Check** | Code that asks whether the expectation is met. | Compare available date pairs. |
| **Outcome** | Passed, Flagged or Untested for this check. | No containment date means date order is Untested. |
| **Finding** | What we observed. | Some records have no containment date. |
| **Issue** | A finding or limitation that needs a decision for our analysis. | We cannot calculate duration for those records. |
| **Evidence** | The records or values supporting the finding. | Original affected rows, with their source-row labels. |

| Origin of an expectation | Example | Important qualification |
| --- | --- | --- |
| Dataset documentation | Fourth-edition coverage is 1992–2015. | That period belongs to this source edition. |
| Analysis question | Q3 needs both dates. | Missing containment can be permitted in the source and still limit Q3. |
| Meaning and chronology | Containment should not be earlier than discovery. | A flag still needs source review. |
| Classroom convention | Compare present names without outside spaces. | This is our formatting expectation, not proof of incorrect fire identity. |
| Classroom plausibility rule | Review non-positive recorded area. | Review the source measurement before correcting or removing anything. |

**References:** [Source record](https://www.fs.usda.gov/rds/archive/catalog/RDS-2013-0009.4) · [Source field metadata](https://www.fs.usda.gov/rds/archive/products/RDS-2013-0009.4/_metadata_RDS-2013-0009.4.html).
Postal code spellings can be checked against [USPS Publication 28, Appendix B](https://pe.usps.com/text/pub28/28apb.htm).
The supplied state-code list deliberately covers the 50 states, DC and PR for this teaching case;
it is not a complete list of every postal abbreviation or a rule for every dataset.

**One numbering system:** `CHK01`–`CHK14` are fixed check references. The digits do not indicate
severity or outcome. A check keeps its reference when it passes. Its issue row, if required,
uses the same reference. `Outputs/evidence/CHK02.csv` contains the evidence for `CHK02`.
One issue row can be supported by many evidence rows. We do not assign a new issue number
to every affected record.

The next table is the complete check catalogue for this introductory guide. Read the origin
and prerequisite as well as the expectation. Each check is implemented once in its named chapter.


In [ ]:
# This is a small reference table, not a new inspection of the source records.
check_catalogue = pd.read_csv("Check_Catalogue.csv")
display(check_catalogue[["Check ID", "Chapter", "Check", "Expectation", "Prerequisite"]])


**Student activity:** Explain the origin of the 1992–2015 period and the need for both dates.
**Suggested reasoning:** The period comes from the selected source edition. Both dates are
needed because of our duration question. Neither expectation was chosen by looking for a value
that would make the check pass.


## Block 6 · Distinguish Passed, Flagged and Untested

**Objective:** Decide whether a record meets a rule, needs review, or cannot yet be checked.

**Rule for this demonstration:** Containment is on or after discovery. Both dates must be usable.

| Record outcome | Meaning | Handling |
| --- | --- | --- |
| **Passed** | We could test this rule and found no violation. | Record the result. Passing this rule does not establish that every field is correct. |
| **Flagged** | We could test this rule and found a condition needing review. | Keep evidence, explain the effect and investigate before treatment. |
| **Untested** | Required information was unavailable or unsuitable. | State why, preserve the limitation and investigate recovery or an appropriate analysis subset. |

The following outcomes are written by hand to teach their meaning. The real date code is
introduced in Chapters 3 and 4. These are invented examples, not findings from `raw`.


In [ ]:
outcome_demo = pd.DataFrame({
    "Record": ["A", "B", "C", "D"],
    "Discovery": ["10 July", "10 July", "10 July", "Missing"],
    "Containment": ["12 July", "9 July", "Missing", "12 July"],
    "Date-order outcome": ["Passed", "Flagged", "Untested", "Untested"]
})
display(outcome_demo)

# Count the three disjoint outcomes; each demonstration record belongs to one.
demo_outcomes = outcome_demo["Date-order outcome"]
demo_passed = (demo_outcomes == "Passed").sum()
demo_flagged = (demo_outcomes == "Flagged").sum()
demo_untested = (demo_outcomes == "Untested").sum()
print("Total:", len(outcome_demo))
print("Passed:", demo_passed, "Flagged:", demo_flagged, "Untested:", demo_untested)
print("Checked:", demo_passed + demo_flagged)


**Read the result:** Four records = one Passed + one Flagged + two Untested.
Two records were checked: the Passed record and the Flagged record.

**Count relationships:** Total = Passed + Flagged + Untested. Checked = Passed + Flagged.
Do not add Checked and Flagged: Flagged is already part of Checked.

**A check summary:** If any checked records are Flagged, the check result is Flagged.
If no records can be checked, it is Untested. Otherwise it is Passed **for checked records**.
Always display the Untested count too. A zero flagged count does not establish complete coverage.

**One record, different checks:** Record C is Flagged by a containment-availability check
because the date is absent. It is Untested by date order because we cannot compare the dates.
Both outcomes are appropriate. Outcomes belong to checks, not permanently to the whole record.

**Student activity:** If 100 records contain 90 Passed, zero Flagged and ten Untested,
what can we conclude? **Suggested answer:** Date order passed for 90 checked records.
Ten records still could not be tested; they do not become Passed.


## Block 7 · Connect a result to evidence and a decision

**Objective:** Understand the three reporting tables before creating them in Chapter 2.

| Resource | Meaning of one row | Purpose |
| --- | --- | --- |
| Check log | One defined check | Shows what was tested, counts and outcome, including passed checks. |
| Issue log | One concern or limitation for a check | Explains the finding, its effect and the proposed next action. |
| Evidence | One affected source record in this guide | Shows the original values and any comparison values needed to explain the concern. |

Record B supports one date-order issue. Its evidence is the original pair of dates.
Records C and D support the limitation that not all records have usable date pairs.
We can propose investigating the dates; that proposal does not carry out cleaning.

**Counting discipline:** A record may support several findings. Adding issue counts does not
give a count of distinct problematic records. `source_row` links evidence to the row label in
this unchanged Pandas import; it is not an incident ID or a physical CSV line number.

**Study task:** For the four-record example, write one sentence stating the duration limitation
and one proposed action. **Suggested response:** Two records lack a date needed for comparison;
one available pair has reversed order. Investigate the unavailable and reversed dates before
choosing the records to include in duration analysis.

**Chapter recap:** You should now be able to explain the dataset, distinguish table/column/condition/count,
justify a rule and distinguish the three outcomes. Chapter 2 applies this process to missing
information and exact repeated records.

**Quick glossary:** DataFrame = labelled table. Series = labelled sequence, often a column.
Boolean = True/False. Index = row labels. Prerequisite = information needed before a particular
check can be performed. Conversion = interpreting a value as another type. A preview shows examples;
a full-data count measures the complete population stated for the check.
